# DWthon: RAG pod lupą — dzień 3

## Od metadanych do Mapy Wiedzy: vault Obsidiana i brama do inteligentnego retrievalu

Dzień 3. **Finał.**

Jeśli udało się dojść tutaj — gratulacje. Niewielki procent osób dochodzi do końca 3-dniowych maratonów. Przypomnijmy szybko, co już mamy w rękach:

- **Dzień 1:** setup (GitHub, Obsidian, połączenie z Bielikiem). Zrozumienie, że RAG to nie „wrzuć PDF do bazy wektorowej" — to **system**.
- **Dzień 2:** `Structured Output` w praktyce. Klasyfikator dokumentów, klasyfikator tabel. Zobaczyliśmy, dlaczego Poziom 2 jest fundamentem wszystkiego, co dzieje się wyżej.

Dzisiaj **składamy to w artefakt, który zostaje z Tobą**.

### Czego nauczymy się dzisiaj (Dzień 3)

1. Zobaczymy, czym **tak naprawdę** jest Mapa Wiedzy — i dlaczego to nie jest to samo, co „graf w Obsidianie"
2. Z metadanych **1417 tabel** (wygenerowanych Twoim klasyfikatorem z Dnia 2) zbudujemy **vault Obsidiana**
3. Obejrzymy **graf powiązań** — 7 firm, warstwa konceptów, mosty cross-company
4. Porozmawiamy, **jak to się ma do skali** (tysiące vs miliony dokumentów) — i gdzie leży granica sensowności Obsidiana
5. W zadaniu zaliczeniowym **zaprojektujesz pierwszy element retrievalu** — Pydantic-ową strukturę, która rozkłada pytanie użytkownika na parametry
6. Zapiszemy całą Mapę Wiedzy w Twoim repo na GitHubie
7. To **zostaje z Tobą na zawsze** — edytowalne, rozszerzalne, Twoje

<img src="../images/graf_example.jpg" width="900px" />

*Tak będzie wyglądać Twój graf Mapy Wiedzy w Obsidianie za ~30 minut. 7 „wysp" (firm), połączonych cienkimi mostami przez warstwę konceptów (obszarów finansowych).*

## Ważna uwaga techniczna (i organizacyjna)

Żeby zmieścić się w trzech dniach i nie topić Cię w kodzie, **przygotowałem już wszystkie dane wejściowe**:

- Przerobiłem 7 sprawozdań finansowych (PDF → markdown → tabele)
- Uruchomiłem klasyfikatory z Dnia 2 na **1417 tabelach** (to zajęło trochę czasu i pochłonęło trochę tokenów)
- Zapisałem wynik w jednym pliku parquet, który wczytamy za chwilę

**Czego świadomie nie zrobimy dzisiaj:**

- Nie będziemy implementować pełnego retrievalu „w pandas" krok po kroku
- Nie będziemy odpalać Gradio ani żadnego UI

Dlaczego? Bo w 3 dni nie da się rzetelnie pokazać **wszystkiego**. Retrieval to osobna, duża dyscyplina — hybryda BM25 + dense + re-ranking, ewaluacja, pytania pomocnicze, dekompozycja, obsługa błędów, monitoring. W [Praktycznym LLM](https://dataworkshop.eu/pl/praktyczny-llm?utm_source=dwthon&utm_medium=lesson&utm_campaign=rag&utm_term=day3) poświęcamy na to **cały osobny moduł (a cały kurs to 5 tygodni)**. Tutaj byłaby to tylko namiastka, która wprowadziłaby więcej zamieszania niż jasności.

Zamiast tego **skupimy się na tym, co naprawdę fundamentalne**: na reprezentacji wiedzy (Mapie) i na **pierwszym, najważniejszym ogniwie** retrievalu — zrozumieniu, co użytkownik w ogóle pyta. To ogniwo zaprojektujesz sam/a w zadaniu zaliczeniowym.


## Wczytujemy biblioteki

Tak jak wczoraj — kod nie gryzie. Jeśli nie programujesz na co dzień, po prostu klikaj, uruchamiaj, obserwuj wynik.

In [ ]:
import enum
from pathlib import Path
from datetime import date
from typing import List, Optional

import pandas as pd
from pydantic import BaseModel, Field
from openai import OpenAI
from dotenv import load_dotenv
import instructor

import subprocess
from helper import build_vault

## Inicjalizacja klienta

In [ ]:
load_dotenv()

client = OpenAI(base_url="https://llm-api.dataworkshop.eu")
client = instructor.patch(client, mode=instructor.Mode.MD_JSON)

## Krok 1: Wczytujemy przygotowane dane

Mamy jeden plik `tables.parquet`, który zawiera **wszystko, co potrzebne** — scalone metadane dokumentów, tabele i klasyfikacje LLM z Dnia 2. Po jednym rekordzie na każdą tabelę.

In [ ]:
df = pd.read_parquet("tables.parquet")

print(f"📊 Tabel:       {len(df)}")
print(f"📄 Dokumentów:  {df['document_id'].nunique()}")
print(f"🏢 Firm:        {df['entity_name'].nunique()}")
print(f"\nKolumny ({len(df.columns)}):")
for col in df.columns:
    print(f"  - {col}")

### Co mamy w jednym rekordzie?

Każdy wiersz to **jedna tabela** plus pełny kontekst biznesowy (firma, okres, kategoria) plus klasyfikacja LLM (tytuł, opis, tagi, pytania, wnioski). Zobaczmy:

In [ ]:
sample = df.sample(1, random_state=7).iloc[0]

print(f"🏢 Firma:        {sample['entity_name']} {sample['entity_type']}")
print(f"📅 Okres:        {sample['period_start']} → {sample['period_end']}")
print(f"💰 Mnożnik:      {sample['value_multiplier']} PLN")
print(f"\n📊 Tabela:       {sample['title']}")
print(f"🏷️  Tagi:         {list(sample['tags'])[:6]}")
print(f"💱 Waluta:       {sample['currency']}")
print(f"\n📝 Opis:\n{sample['description']}")
print(f"\n❓ Pytania analityczne:")
for q in list(sample['questions'])[:3]:
    print(f"   - {q}")


Zwróć uwagę na to, co mamy dla **każdej** z 1417 tabel:

- **Tytuł** — co to za tabela
- **Opis** — o czym jest (zrozumiały nawet dla osoby nietechnicznej)
- **Tagi** — do szybkiego filtrowania
- **Pytania analityczne** — na jakie pytania tabela odpowiada (złoto dla retrievalu!)
- **Wnioski, zastosowania, waluta, kolumny…** — pełny kontekst

To jest **nasza tarcza przeciwko chaosowi**. Gdy użytkownik zapyta *„Jakie jest zadłużenie Enei?"* — nie musimy przeszukiwać surowego tekstu 300-stronicowego PDF-a. Mamy wprost wskazane: "ta tabela odpowiada na pytania o zadłużenie".

To **nie jest** tekst wygenerowany losowo. To tekst wygenerowany **w rygorze kontraktu** (Pydantic), z walidacją, powtarzalnie. Każdy z tych atrybutów można zweryfikować ręcznie — i to jest pierwszy filar, o którym mówiliśmy przez 3 dni: **audytowalność**.


## Krok 2: Mapa Wiedzy jako vault Obsidiana

Zanim wygenerujemy vault — zatrzymajmy się na chwilę. To jest moment, w którym **łatwo źle zrozumieć, co robimy**.

### Karpathy vs nasze podejście

Karpathy w swojej „LLM Wiki" zaproponował, żeby **LLM generował całą wiki samodzielnie**. Dajesz mu surowe źródła, on pisze artykuły, buduje linki, sam decyduje o strukturze. Ładne demo — ale ma co najmniej trzy problemy, które warto sobie nazwać:

1. **Koszty.** Tokeny wyjściowe są **najdroższe**. Generowanie 1500 artykułów przez LLM to rachunek na kilkadziesiąt dolarów. Dla jednej firmy, jednej domeny. Skala się sypie natychmiast.
2. **Kontrola.** Gdy LLM pisze artykuł „z niczego", masz minimalny wpływ na to, co się w nim pojawi. Nie masz spójnego formatu. Każdy artykuł ma inną strukturę. **Audyt jest trudny.**
3. **Powtarzalność.** Uruchom dwa razy — dostaniesz dwa różne vaulty. W produkcji to koszmar utrzymaniowy.

### Nasze podejście: szablony + LLM w rygorze

My robimy to **sprytniej**:

- **Szablony** (deterministyczne) definiują **strukturę** — jakie sekcje, jakie linki, jaki układ. To zwykły kod Python w `helper.py`.
- **LLM** (z Dnia 2) wygenerował **treść metadanych** — tytuł, opis, tagi, pytania. Raz, offline, w rygorze Structured Output.
- Przy generowaniu vaultu **tylko składamy** to razem. Zero dodatkowych tokenów.

Efekt? **Pełna kontrola nad strukturą, zero kosztów przy generowaniu, 100% powtarzalność.** A treść artykułów i tak jest wygenerowana przez LLM — tylko raz, na etapie klasyfikacji.

> ☝️ Co najważniejsze — **tylko wyobraźnia ogranicza, jakie szablony możesz zbudować**. Sektory per branża? Timeline zmian wskaźników? Mapy ryzyka? To wszystko to tylko inna funkcja w `build_vault()`. Treść już mamy.


### Struktura vaultu, który za chwilę wygenerujemy

```
dwthon-rag/
└── vault/
    ├── 00-MOC/                  ← mapy nawigacyjne (Maps of Content)
    │   ├── Baza Wiedzy.md       ← główny index
    │   ├── Energetyka (sektor).md
    │   ├── Bankowość (sektor).md
    │   └── …
    ├── 10-Koncepty/             ← obszary cross-company (MOSTY między firmami)
    │   ├── Bilans.md
    │   ├── Ryzyko.md
    │   ├── Rachunek Zysków i Strat.md
    │   └── …
    └── 20-Firmy/
        ├── Enea/
        │   ├── Enea.md
        │   ├── Obszary/         ← huby tematyczne per firma
        │   │   ├── Enea — Bilans.md
        │   │   ├── Enea — Ryzyko.md
        │   │   └── …
        │   └── Tabele/          ← ok. 165 liści
        │       ├── Enea T000 …md
        │       └── …
        └── …
```

### Dlaczego akurat taka?

Chcemy, żeby graf w Obsidianie wyglądał jak **kilka wysp z cienkimi mostami**, a nie jak kula makaronu.

- **Wyspy firm** są ciasne (tabele → hub tematyczny firmy → firma)
- **Mosty cross-company** istnieją **tylko** przez **warstwę konceptów** (ok. 15 notatek)
- Tabela (liść) ma tylko **1 link strukturalny** — do swojego huba tematycznego

To jest projektowanie grafu pod **czytelność i nawigację**, nie pod „więcej linków = lepiej".

> ⚠️ **Uwaga merytoryczna.** To, co za chwilę zrobimy, to **jedna z wielu możliwych reprezentacji** tej samej wiedzy. Akurat ta jest dzisiaj najgłośniejsza (Karpathy, GraphRAG, Obsidian-as-vault). Ale mógłbyś ją zbudować jako: płaski zbiór tabel w bazie, graf w Neo4j, dedykowany index w Elasticu, albo hybrydę tego wszystkiego. **Wybór reprezentacji zawsze zależy od pytań, jakie chcesz zadawać.** O tym szerzej w Praktycznym LLM.

## Krok 3: Generujemy vault

Cała logika jest w `helper.py`. Nie musisz tego czytać — ale jeśli chcesz, zajrzyj i zobacz, jak działa szablon. Kod jest krótki i czytelny (i *nie zużywa* ani jednego tokena LLM-a — wszystko już mamy).

### Gdzie trafia vault?

Przypomnijmy strukturę z Dnia 1:

```
dwthon/
├── notebooks/
│   └── day3.ipynb    ← tu jesteś
└── dwthon-rag/       ← sklonowane repo z GitHuba (Dzień 1)
    └── (tu trafi vault/)
```

Vault wygenerujemy **bezpośrednio do sklonowanego repo**, żeby za chwilę zrobić commit i push.


<div style="border: 1px solid #f5c2c7; background-color: #f8d7da; color: #842029; padding: 15px; border-radius: 5px; font-family: Arial, sans-serif;">
  <strong>⚠️ Uwaga!</strong>
  <p style="margin: 10px 0 0 0;">
    Ścieżka do Twojego sklonowanego repo.<br>
    (jeśli masz inaczej — zmień nazwę poniżej)
  </p>
</div>

In [ ]:
REPO_DIR = "./dwthon-rag"

### Generujemy vault wewnątrz repo

In [ ]:
build_vault(df, out_dir=f"{REPO_DIR}/vault")

### Sprawdź, co się wygenerowało

In [ ]:
vault_path = Path(REPO_DIR) / "vault"

for path in sorted(vault_path.iterdir()):
    if path.is_dir():
        n_files = sum(1 for _ in path.rglob("*.md"))
        print(f"📁 {path.name}/  ({n_files} plików)")

#### Obejrzyjmy przykładową notatkę — koncept **Ryzyko**, który łączy wszystkie firmy:

In [ ]:
concept_file = vault_path / "10-Koncepty" / "Ryzyko.md"
print(concept_file.read_text(encoding="utf-8")[:1500])

Widzisz, jak wygląda koncept? To jest **most między firmami**. Kiedy analityk pyta *„Porównaj ryzyko kredytowe Enei vs PGE"*, otwiera ten jeden plik i widzi listy obu firm do porównania. Dwa kliki.

I — co ważne — **każdy link w tym pliku prowadzi do konkretnej tabeli z konkretnego dokumentu**. Nie ma halucynacji, nie ma „chyba to było gdzieś w PDF-ie". Jest ścieżka: koncept → firma → tabela → źródło. To jest **audytowalność w praktyce**.


## Krok 4: Push do GitHuba

Dzień 1 ustawił podstawy — masz sklonowane prywatne repo z tokenem zapisanym w git credentials. Teraz robimy standardowy **commit + push**.

### Opcja A: przez terminal (dla bardziej doświadczonych)

Otwórz terminal **na swoim laptopie** (nie w notebook'u!), przejdź do folderu repo i wykonaj:

```bash
cd ~/dwthon/notebooks/dwthon-rag

git add vault/
git commit -m "DWthon Day 3: Mapa Wiedzy — 1417 tabel, 7 spolek"
git push origin main
```

> ⚠️ Jeśli Twoja gałąź to `master` (starsze repo), użyj `git push origin master`.

### Opcja B: z notebook'a

Jeśli notebook działa na serwerze DataWorkshop, terminal laptopa nie widzi tych plików. Wtedy pushuj z notebook'a:

In [ ]:
def run_git(cmd, cwd):
    result = subprocess.run(cmd, cwd=cwd, shell=True, capture_output=True, text=True)
    output = result.stdout.strip() or result.stderr.strip() or "OK"
    print(f"$ {cmd}\n{output}\n")
    return result

run_git("git add vault/", cwd=REPO_DIR)
run_git('git commit -m "DWthon Day 3: Mapa Wiedzy - 1417 tabel, 7 spolek"', cwd=REPO_DIR)
run_git("git push", cwd=REPO_DIR)

<div class="alert alert-block alert-info">
<h4>💡 Ściągnięcie vaultu na laptopa</h4>
<p>Po udanym <code>push</code> na GitHub — żeby otworzyć Mapę w Obsidianie na swoim laptopie, wykonaj:</p>
<pre>cd path/to/dwthon-rag
git pull</pre>
<p>Folder <code>vault/</code> pojawi się lokalnie. Teraz otwórzmy go w Obsidianie.</p>
</div>

## Krok 5: Obsidian — oglądamy graf

Teraz najfajniejsza część. Otwieramy Mapę w Obsidianie i patrzymy na graf.

### Krok po kroku

1. **Otwórz Obsidian** na laptopie
2. Kliknij **„Open folder as vault"** (lub menu → `File` → `Open vault`)
3. Wskaż folder `dwthon-rag/vault` (tylko podfolder `vault/`, nie całe repo)
4. Gdy vault się otworzy — zobaczysz w lewym panelu strukturę `00-MOC`, `10-Koncepty`, `20-Firmy`
5. W **lewym dolnym rogu** kliknij ikonę **grafu** (kula z liniami) — albo użyj skrótu `Ctrl+G` / `Cmd+G`

Na początku zobaczysz coś w stylu: kula makaronu. Nie zniechęcaj się — teraz ją uporządkujemy.

### Ustawienia grafu (żeby ładnie wyglądał)

Po prawej stronie grafu masz panel ustawień. Ustaw:

**Filters:**
- W polu `Search` możesz wpisać np. `-tag:#tabela` aby ukryć 1400 liści i zobaczyć sam **szkielet wiedzy** (firmy + huby + koncepty). Różnica jest kolosalna.

**Groups** (kluczowe dla czytelności):

| Search |
|---|
| `path:"20-Firmy/Enea"` |
| `path:"20-Firmy/PGE"` |
| `path:"20-Firmy/PZU"` |
| `path:"20-Firmy/ORLEN"` |
| `path:"20-Firmy/PKO Bank Polski"` |
| `path:"20-Firmy/TAURON"` |
| `path:"20-Firmy/KGHM"` |
| `tag:#koncept` |
| `tag:#hub` |


<img src="../images/graf_color.jpg" width="1100px" />


**Display:**
- Zwiększ `Text fade threshold` — etykiety będą czytelne z daleka
- Pokaż `Arrows` — żeby widzieć kierunek linków

**Forces** (fizyka grafu):
- `Center force` ok. **0.5** — grupuje wyspy
- `Repel force` ok. **10** — rozsuwa węzły
- `Link force` ok. **1** — trzyma powiązane razem
- `Link distance` ok. **250** — rozciąga graf

Eksperymentuj z suwakami. Cel — zobaczyć 7 wyraźnych wysp (firmy) z cienkimi mostami przechodzącymi przez centralne koncepty.

## Krok 6: Czym **tak naprawdę** jest Mapa Wiedzy?

OK, masz graf w Obsidianie. Wygląda ładnie, klika się. Ale tu jest **pułapka myślowa**, w którą wpada większość osób na etapie „hype-u na GraphRAG":

> **Mapa Wiedzy to nie jest graf w Obsidianie.** Graf w Obsidianie to tylko **jedna z wizualizacji** Mapy.

Mapa Wiedzy to coś znacznie bardziej abstrakcyjnego: **to uporządkowana ścieżka, po której można się po wiedzy nawigować**. Może ją wizualizować Obsidian. Może ją wizualizować graf w Neo4j. Może ją w ogóle nie wizualizować nic — bo „chodzi" po niej kod (retrieval).

### Schemat ścieżki nawigacyjnej

Spójrz na to tak:

```
                  ┌─────────────────────────────┐
                  │   Pytanie użytkownika       │
                  │  "Jakie jest zadłużenie     │
                  │   Enei w 2023?"             │
                  └──────────────┬──────────────┘
                                 │
                                 ▼
         ┌───────────────────────────────────────────┐
         │  [1] PARSOWANIE INTENCJI                  │
         │  Wyciągamy: firma, okres, obszar          │
         │  (to zrobisz w zadaniu zaliczeniowym!)    │
         └──────────────┬────────────────────────────┘
                        │ {firma: "Enea", rok: 2023,
                        │  obszar: "Zadłużenie"}
                        ▼
         ┌───────────────────────────────────────────┐
         │  [2] ZAWĘŻENIE PRZESTRZENI                │
         │  Z 1417 tabel → kilkanaście kandydatów    │
         │  Metoda: zwykłe filtrowanie po metada-    │
         │  nych (pandas / SQL / baza dokumentów)    │
         └──────────────┬────────────────────────────┘
                        │
                        ▼
         ┌───────────────────────────────────────────┐
         │  [3] WYBÓR NAJLEPSZYCH (przez LLM)        │
         │  Krótka lista + opisy → top 1-5           │
         │  Nie wysyłamy surowych tabel, tylko       │
         │  metadane!                                │
         └──────────────┬────────────────────────────┘
                        │
                        ▼
         ┌───────────────────────────────────────────┐
         │  [4] KONTEKST + ATRYBUCJA                 │
         │  Markdown wybranych tabel                 │
         │  + document_id + table_id (źródło!)       │
         └──────────────┬────────────────────────────┘
                        │
                        ▼
         ┌───────────────────────────────────────────┐
         │  [5] ODPOWIEDŹ Z CYTATAMI                 │
         │  LLM + Structured Output                  │
         │  → final_answer + sources + confidence    │
         └───────────────────────────────────────────┘
```

**Obsidian pokazuje Ci krok [0]** — strukturę całości. Tę, po której „chodzi" kod w kolejnych krokach. Możesz ją czytać oczami (otwierasz notatkę, klikasz link, idziesz dalej). Możesz po niej chodzić kodem (filtrujesz metadane, wybierasz kandydatów). **To jest ta sama ścieżka, tylko widoczna z innej strony.**

### Kiedy Obsidian wystarcza, a kiedy już nie

**Przy 1417 tabelach:**
- Obsidian jest genialny — otwierasz, klikasz, znajdujesz w kilka sekund
- Graf pokazuje całość na jednym ekranie
- Analityk biznesowy **może to sam przeglądać** — to jest wartość sama w sobie

**Przy 10-100 tabelach:**
- Obsidian zaczyna się zacinać (zwłaszcza graf)
- Klikanie przestaje skalować się dla człowieka
- Ale struktura (foldery, metadane) nadal jest poprawna

**Przy 1000 lub więcej tabelach:**
- Obsidian się nie nadaje jako wizualizacja całości
- Ale **idea Mapy Wiedzy nadal jest ważna** — tylko nawigacja przenosi się w pełni do kodu

W tym ostatnim przypadku pod spodem siada zwykły **query engine**:

- **SQL** (Postgres, DuckDB, ClickHouse) — idealne do filtrowania po metadanych
- **Pandas** — spokojnie uniesie miliony wierszy na jednej maszynie (w DuckDB jeszcze więcej)
- **Elasticsearch / OpenSearch** — gdy chcemy dodać pełnotekstowy search
- ...

Ale **ani jedno, ani drugie nie zmienia fundamentu** — nadal chodzisz po tej samej ścieżce z diagramu. Zmienia się **warsztat**, nie **idea**.

> 🔑 Wniosek: Mapa Wiedzy to **projekt**, nie narzędzie. Obsidian to **jedna z soczewek** na ten projekt — bardzo wygodna dla człowieka na małej/średniej skali. W produkcji projekt zostaje, soczewka się zmienia.

## Krok 7: Brama do retrievalu — parsowanie intencji

Cały pipeline z diagramu zaczyna się od jednego, kluczowego kroku: **zrozumienia, co użytkownik w ogóle pyta**.

Jeśli tego kroku nie zrobisz dobrze — **cała reszta jest stracona**. Nie ma znaczenia, jak świetny masz retrieval, jak precyzyjne metadane, jaki drogi model do generacji odpowiedzi. Jeśli system nie rozumie, że *„wyniki PKO w zeszłym roku"* to `firma=PKOBP, rok=2023` — to nigdy nie doleci do właściwej tabeli.

I tu wraca wszystko, co robiliśmy przez 3 dni:

- **Poziom 2 (Structured Output)** — kontrakt, który zmusza LLM do oddania konkretnych pól
- **Pydantic** — walidacja, typy, enum-y, opisy
- **Audytowalność** — każdy krok da się podejrzeć (co LLM wyciągnął, co filtr zwrócił, co LLM wybrał)

**To jest moment, w którym wszystko z Dnia 1 i Dnia 2 spotyka się w jednym miejscu.** I to zaprojektujesz teraz samodzielnie.

## 🧠 Zadanie zaliczeniowe (Dzień 3): Parser intencji użytkownika

Pamiętasz diagram retrievalu sprzed chwili? Krok **[1]** to **parser intencji** — serce całego systemu. Dostaje na wejściu pytanie w języku naturalnym, zwraca **ustrukturyzowane parametry** (firma + okres), którymi potem można przefiltrować zbiór tabel.

> To są dokładnie te 20-30 linijek kodu, które **decydują o jakości całego RAG-u**. Słaby parser = słaby retrieval = słaba odpowiedź. Silny parser = reszta prawie robi się sama.

Zadanie składa się z **3 etapów**. Każdy ma jasny cel, podpowiedź i (gdyby utknąć) rozwiązanie do podejrzenia. Polecam jednak spróbować samodzielnie — właśnie teraz domyka się intuicja z Dnia 2 (`Structured Output`).

### 🎯 Etap 1/3: Zdefiniuj strukturę danych (Pydantic)

**Co masz zrobić:**

Zaprojektuj **3 struktury**, które będą opisywać sparsowane pytanie:

1. **`Company`** — enum z **7 firmami** z naszego zbioru. Nazwy firm weź z danych:
   ```python
   df["entity_name"].unique()
   ```
2. **`COMPANY_DESCRIPTIONS`** — słownik `Company → str`, z krótkim opisem biznesowym każdej firmy (branża, czym się zajmuje). **To jest ważne** — ten słownik trafi do promptu jako kontekst dla LLM-a, żeby wiedział, że *„PKO BP"*, *„PKO S.A."* i *„PKO Bank Polski"* to ta sama firma.
3. **`TimePeriod`** — model z polami `start_date: date` i `end_date: date`.
4. **`UserQuestion`** — spina oryginalne pytanie + listę firm + okres.

**Wymagania techniczne:**
- Każde pole musi mieć `Field(..., description="...")` — LLM **czyta opisy pól** i kieruje się nimi przy generowaniu odpowiedzi (Dzień 2!).
- `UserQuestion.company` to **lista** (bo pytanie może dotyczyć kilku firm).

<details>
<p>    
<summary style="background: #fff3cd; padding: 6px; border-left: 4px solid #ffc107;">
💡 <b>Podpowiedź — struktura szkieletu</b>
</summary>

```python
import enum
from datetime import date
from typing import List
from pydantic import BaseModel, Field

class Company(enum.Enum):
    # TODO: wpisz wszystkie 7 firm — nazwy weź z df["entity_name"].unique()
    ENEA = 1
    # ...

COMPANY_DESCRIPTIONS = {
    # TODO: opis każdej firmy — branża + jedno zdanie kontekstu
    Company.ENEA: "Enea S.A. — polska grupa energetyczna.",
    # ...
}

class TimePeriod(BaseModel):
    start_date: date = Field(..., description="...")
    end_date:   date = Field(..., description="...")

class UserQuestion(BaseModel):
    original_question: str = Field(..., description="...")
    company: List[Company] = Field(..., description="...")
    time_period: TimePeriod = Field(..., description="...")
```

Zwróć uwagę, że w opisie pola `company` warto **wprost powiedzieć LLM-owi**, co zrobić, jeśli firma nie jest podana (zwrócić wszystkie — żeby nie wykluczać kandydatów).



<details>
    
<summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 18px; font-weight: 900;">
✅ Rozwiązanie Etapu 1
</summary>

```python
import enum
from datetime import date
from typing import List
from pydantic import BaseModel, Field

class Company(enum.Enum):
    ENEA    = 1
    TAURON  = 2
    PKOBP   = 3
    KGHM    = 4
    PGE     = 5
    ORLEN   = 6
    PZU     = 7

COMPANY_DESCRIPTIONS = {
    Company.ENEA:   "Enea S.A. — polska grupa energetyczna (wytwarzanie, dystrybucja, obrót).",
    Company.TAURON: "TAURON Polska Energia S.A. — dystrybucja i sprzedaż energii elektrycznej.",
    Company.PKOBP:  "PKO Bank Polski (PKO BP, PKO S.A.) — największy bank w Polsce.",
    Company.KGHM:   "KGHM Polska Miedź S.A. — globalny producent miedzi i srebra.",
    Company.PGE:    "PGE (Polska Grupa Energetyczna) — największa grupa energetyczna w Polsce.",
    Company.ORLEN:  "PKN Orlen — koncern multienergetyczny (paliwa, petrochemia, energetyka).",
    Company.PZU:    "PZU (Powszechny Zakład Ubezpieczeń) — wiodąca grupa ubezpieczeniowa w Polsce.",
}

class TimePeriod(BaseModel):
    start_date: date = Field(..., description="Data początkowa okresu sprawozdawczego (YYYY-MM-DD).")
    end_date:   date = Field(..., description="Data końcowa okresu sprawozdawczego (YYYY-MM-DD).")

class UserQuestion(BaseModel):
    original_question: str = Field(..., description="Oryginalne pytanie użytkownika — bez zmian.")
    company: List[Company] = Field(
        ...,
        description=(
            "Lista firm wymienionych w pytaniu. Jeśli pytanie nie wskazuje konkretnej firmy — "
            "ZWRÓĆ WSZYSTKIE dostępne firmy (nie wykluczaj żadnej). Słownik firm: "
            + ", ".join(f"{k.name}: {v}" for k, v in COMPANY_DESCRIPTIONS.items())
        )
    )
    time_period: TimePeriod = Field(
        ...,
        description="Okres sprawozdawczy, którego dotyczy pytanie."
    )
```

</details>

</details>

In [ ]:
## YOUR CODE HERE 1/3

### 🎯 Etap 2/3: Napisz funkcję `parse_question`

**Co masz zrobić:**

Napisz funkcję, która przyjmuje pytanie (string) i zwraca obiekt `UserQuestion`, korzystając z `instructor`-a i `response_model` (dokładnie jak wczoraj — ten sam wzorzec co klasyfikator tabel).

**Wymagania dla promptu systemowego (to jest najważniejsza część!):**

Prompt **musi jasno odpowiedzieć na 4 pytania**, bo inaczej LLM zacznie zgadywać:

| Sytuacja | Co LLM ma zrobić |
|---|---|
| Firma nie jest podana w pytaniu | Zwróć **wszystkie** (inaczej wykluczysz kandydatów na wstępie) |
| Okres nie jest podany | Fallback: cały ostatni zamknięty rok (np. dziś `2024-11-20` → `2023-01-01` do `2023-12-31`) |
| Okres podany niepełnie (*"w 2023"*) | Rozszerz do pełnego roku: `2023-01-01` do `2023-12-31` |
| Okres względny (*"zeszły rok", "ostatni kwartał"*) | Policz względem dzisiejszej daty |

**Wskazówka:** przekaż LLM-owi `date.today()` w prompcie — inaczej model nie wie, który jest „zeszły rok".

<details>
<summary style="background: #fff3cd; padding: 6px; border-left: 4px solid #ffc107;">
💡 <b>Podpowiedź — szkielet funkcji</b>
</summary>

```python
def parse_question(question: str, model="gpt-4o-mini") -> UserQuestion:
    system_prompt = f"""
Jesteś ekspertem od sprawozdań finansowych polskich spółek giełdowych.
Twoim zadaniem jest WYCIĄGNĄĆ z pytania: (1) firmę/firmy, (2) okres sprawozdawczy.

Dzisiejsza data: {date.today()}

ZASADY:
- Jeśli w pytaniu JEST firma → zwróć tylko ją.
- Jeśli NIE MA firmy → ...                           # TODO
- Jeśli NIE MA okresu → ...                          # TODO
- Jeśli okres podany częściowo ("w 2023") → ...      # TODO
- Jeśli okres względny ("zeszły rok") → ...          # TODO
"""
    return client.chat.completions.create(
        model=model,
        response_model=UserQuestion,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": question},
        ],
    )
```



<details>
<summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 18px; font-weight: 900;">
✅ Rozwiązanie Etapu 2
</summary>

```python
def parse_question(question: str, model="gpt-4o-mini") -> UserQuestion:
    system_prompt = f"""
Jesteś ekspertem od sprawozdań finansowych polskich spółek giełdowych.
Twoim zadaniem jest wyciągnąć z pytania użytkownika: (1) firmę/firmy, (2) okres sprawozdawczy.

Dzisiejsza data: {date.today()}

ZASADY DOTYCZĄCE FIRMY:
- Jeśli w pytaniu jest wyraźnie wskazana firma (np. "ORLEN", "PKO BP", "Enea") — zwróć tylko ją.
- Jeśli pytanie porównuje kilka firm ("porównaj Eneę i PGE") — zwróć wszystkie wymienione.
- Jeśli pytanie NIE wskazuje żadnej firmy ("które spółki mają...", "porównaj energetyków") —
  ZWRÓĆ WSZYSTKIE 7 firm. Nie próbuj zgadywać. Lepiej zwrócić za dużo niż wykluczyć kandydatów.

ZASADY DOTYCZĄCE OKRESU:
- Jeśli w pytaniu nie ma okresu — użyj całego POPRZEDNIEGO zamkniętego roku
  (np. dziś jest 2024-11-20 → zwróć 2023-01-01 do 2023-12-31).
- Jeśli okres jest podany częściowo ("w 2023") — rozszerz do pełnego roku
  (2023-01-01 do 2023-12-31).
- Jeśli okres jest względny ("zeszły rok", "ostatni rok", "w zeszłym roku") —
  policz go względem dzisiejszej daty.
- Jeśli pytanie wspomina kwartał ("I kwartał 2023") — zwróć odpowiednie 3 miesiące
  (I kwartał 2023 = 2023-01-01 do 2023-03-31).
"""
    return client.chat.completions.create(
        model=model,
        response_model=UserQuestion,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": question},
        ],
    )
```

</details>

</details>

In [ ]:
## YOUR CODE HERE 2/3

### 🎯 Etap 3/3: Przetestuj parser na 5 pytaniach

**Co masz zrobić:**

Uruchom `parse_question` na **5 pytaniach z listy poniżej** i dla każdego wypisz, co LLM wyciągnął. Przynajmniej **dwa pytania** są celowo podchwytliwe (brak firmy / brak okresu / okres niepełny / okres względny). Sprawdź, czy parser sobie radzi.

```python
test_questions = [
    "Jakie były przychody ORLEN w 2023?",
    "Porównaj wyniki Enei i PGE za zeszły rok",
    "Jakie jest zadłużenie PKO BP?",                 # brak okresu
    "Jakie spółki mają najwyższe ryzyko walutowe?",  # brak firmy
    "Pokaż rentowność TAURON w I kwartale 2023",    # częściowy okres
]
```

**Dla każdego pytania wypisz:**
- listę firm (`[c.name for c in parsed.company]`)
- okres (`start_date → end_date`)
- krótki komentarz: **czy to ma sens biznesowy?** (1 zdanie)

<details>
<summary style="background: #fff3cd; padding: 6px; border-left: 4px solid #ffc107;">
💡 <b>Podpowiedź — jak wypisać wynik</b>
</summary>

```python
for q in test_questions:
    parsed = parse_question(q)
    print(f"❓ {q}")
    print(f"   Firmy: {[c.name for c in parsed.company]}")
    print(f"   Okres: {parsed.time_period.start_date} → {parsed.time_period.end_date}")
    print()
```

Na co zwrócić uwagę przy analizie:
- Czy przy *„zeszły rok"* LLM policzył okres względem `date.today()`?
- Czy przy *„Jakie spółki..."* zwrócił **wszystkie** 7 firm, czy błędnie wybrał jedną?
- Czy *„I kwartał 2023"* = `2023-01-01 → 2023-03-31`?



<details>
<summary style="background: #e6eaeb; padding: 4px 0; text-align: center; font-size: 18px; font-weight: 900;">
✅ Rozwiązanie Etapu 3
</summary>

```python
test_questions = [
    "Jakie były przychody ORLEN w 2023?",
    "Porównaj wyniki Enei i PGE za zeszły rok",
    "Jakie jest zadłużenie PKO BP?",
    "Jakie spółki mają najwyższe ryzyko walutowe?",
    "Pokaż rentowność TAURON w I kwartale 2023",
]

for q in test_questions:
    parsed = parse_question(q)
    companies = [c.name for c in parsed.company]
    period = f"{parsed.time_period.start_date} → {parsed.time_period.end_date}"
    print(f"❓ {q}")
    print(f"   🏢 Firmy ({len(companies)}): {companies}")
    print(f"   📅 Okres: {period}")
    print()
```

**Oczekiwane wyniki (przybliżone):**

| # | Pytanie | Firmy | Okres | Komentarz |
|---|---|---|---|---|
| 1 | "przychody ORLEN w 2023" | `[ORLEN]` | `2023-01-01 → 2023-12-31` | ✅ Firma jasna, rok rozszerzony do pełnego |
| 2 | "Enei i PGE za zeszły rok" | `[ENEA, PGE]` | zależy od `date.today()` | ✅ Dwie firmy, okres względny |
| 3 | "zadłużenie PKO BP" | `[PKO BP]` | fallback (ostatni rok) | ✅ Firma jest, okres z fallbacku |
| 4 | "które spółki..." | `[wszystkie 7]` | fallback | ✅ Brak firmy → zwraca wszystkie |
| 5 | "TAURON I kwartał 2023" | `[TAURON]` | `2023-01-01 → 2023-03-31` | ✅ Kwartał rozwinięty do 3 miesięcy |

Jeśli któryś przypadek **nie wychodzi** — to nie jest błąd Pydantica, tylko **niejasność w prompcie**. Wróć do Etapu 2, dopisz brakującą regułę w `system_prompt` i uruchom ponownie. To jest dokładnie ta iteracyjna praca nad promptem, o której mówiliśmy — kształtowanie zachowania LLM-a przez jasne instrukcje.

</details>

</details>

In [ ]:
## YOUR CODE HERE 3/3

<div class="alert alert-block alert-warning">
<h3>🎁 Bonus +250 punktów — tylko dzisiaj!</h3>
<p><b>To ostatni dzień DWthonu.</b> Za wykonanie zadania z Dnia 3 (i wrzucenie rozwiązania na kanał <code>#dwthon_llm_day3_done</code>) dostajesz:</p>
<ul>
  <li>🏆 <b>Imienny certyfikat ukończenia DWthonu</b> — tylko osoby, które zaliczyły wszystkie 3 dni</li>
  <li>⭐ <b>+250 punktów</b> DataWorkshop — Punkty możesz wykorzystać przy dołączeniu do Praktyczny LLM (poziom 2-3) lub Agentic AI (poziom 4-5).</li>
  <li>🧠 <b>Intuicję, której nie kupisz</b> — zrozumienie, dlaczego parser intencji to najważniejszy kawałek RAG-u</li>
</ul>
<p>Skoro to czytasz i jesteś tutaj — <b>dowieź finał</b>. Pewnie ~15-30 minut pracy dzieli Cię od certyfikatu.</p>
</div>

<div class="alert alert-block alert-danger">
<h2>✅ Zaliczenie Dnia 3 — ostatni dzień!</h2>
<h4>Co masz dostarczyć:</h4>
<ul>
    <li>💻 <b>Kod parsera intencji</b> + wyniki testów na 5+ pytaniach (screenshot lub wklejony output)</li>
    <li>✍️ <b>Krótki komentarz (2-3 zdania):</b> gdzie parser sobie poradził, a gdzie widzisz miejsce na poprawę</li>
    <li>📸 <b>Super, jak też pokażesz screenshot grafu z Obsidiana</b> (po skonfigurowaniu kolorów — Twoje 7 wysp + koncepty)</li>
</ul>
<h4>Gdzie wrzucać?</h4>
<ul>
    <li>Rozwiązania → kanał <a href="https://dataworkshopclub.slack.com/archives/C0AD3JALT3P"><code>#dwthon_llm_day3_done</code></a></li>
    <li>Pytania i wsparcie → <code>#dwthon_llm_day3</code></li>
</ul>
</div>

## 🏆 Certyfikat ukończenia DWthon

Jeśli udało się dowieźć **wszystkie 3 dni** — należy Ci się **imienny certyfikat ukończenia DWthon: RAG pod lupą**.

Bot sprawdza wpisy na kanałach `_done`. Certyfikat dostaniesz (jak masz wszystko zrobione) w poniedziałek.

### ⏰ Dostęp - dzisiaj ostatni dzień

DWthon kończy się dzisiaj. Jutro notebooki i dostęp do Laba zostają zamknięte. Jeśli:

- chcesz **dokończyć zadania w swoim tempie**,
- chcesz **pogrzebać w kodzie i lepiej zrozumieć, jak z parsera wyrasta pełny retrieval**,
- chcesz **mieć dostęp do środowiska do 21 maja 2026**,

→ możesz przedłużyć dostęp przez opcję **DWthon Premium**.

## 🔗 [Przedłuż dostęp → DWthon Premium](https://pay.dataworkshop.eu/?cart=%7B%22products%22%3A%7B%22206%22%3A1%7D%7D&lang=pl&currency=pln&gate=pl)


## 📢 Podziel się ukończonym wyzwaniem

Finał 3-dniowego maratonu — niewiele osób dochodzi do końca. **Zrób z tego hałas:**

1. Tu jest przykładowy baner, możesz użyć go jako szablon lub dodać któryś *(grafika [#1](https://storage.googleapis.com/dw-cert-public/events/dwthon-rag/1.png), [#2](https://storage.googleapis.com/dw-cert-public/events/dwthon-rag/2.png) lub [#3](https://storage.googleapis.com/dw-cert-public/events/dwthon-rag/3.png))* 
2. Krótki post na LinkedIn — oznacz **@Vladimir Alekseichenko** i **@DataWorkshop**
3. Napisz, co najbardziej zaskoczyło w RAG-u — graf w Obsidianie? Że Mapa ≠ graf? A może to, że `Structured Output` nagle „wyjaśnia cały RAG"?

## Podsumowanie Dnia 3 — i całego DWthonu

Trzy dni temu zaczęliśmy od hype'u:

*„Karpathy wyrzucił RAG! Baza wektorowa umarła! LangChain do lamusa!"*

Dziś wiadomo, że to nie był hype. To była **korekta**. Rynek odkrył to, o czym w DataWorkshop mówimy od 2 lat:

> **RAG to nie baza wektorowa. RAG to system reprezentacji wiedzy + inteligentne wyszukiwanie.**

### Co mamy na koniec DWthonu

| Artefakt | Co daje |
|---|---|
| **Mapa Wiedzy** (1417 tabel, 7 firm) | Audytowalna, edytowalna, Twoja |
| **Graf w Obsidianie** | Widzisz powiązania i mosty cross-company |
| **Parser intencji** (Twoje zadanie) | Pierwsze ogniwo retrievalu — działa u Ciebie |
| **Repo na GitHubie** | Zostaje z Tobą, rozwijaj dalej |
| **Intuicja, co to jest "reprezentacja wiedzy"** | Najtrudniejsze do zdobycia — a teraz masz |

### Co najważniejsze

Masz teraz **intuicję**, która odróżnia hype od fundamentów. Gdy jutro pojawi się kolejna moda — "nowa magiczna biblioteka", "rewolucyjny model", "RAG 2.0" — będziesz w stanie w kilka minut ocenić, czy to faktyczna nowość, czy stary zdrowy rozsądek w nowym opakowaniu.

To jest zdolność, której **nie kupisz na Udemy za 39 zł**. Tego trzeba dotknąć.

## Co dalej?

W ciągu 3 dni dotknęliśmy:

- **Poziomu 2** (Structured Output) — fundamentu wszystkiego
- **Mapy Wiedzy** jako reprezentacji danych
- **Vaultu Obsidiana** jako jednej z soczewek na Mapę
- **Pierwszego ogniwa retrievalu** (parser intencji) — sam/a zaprojektowałeś/aś

Świadomie **nie dotknęliśmy** reszty pipeline'u retrievalu (kandydaci, ranking, kontekst, generacja, ewaluacja). Nie dlatego, że to nieważne — dokładnie przeciwnie. Te tematy są na tyle głębokie, że w 3 dni zrobilibyśmy je źle. W [Praktycznym LLM](https://dataworkshop.eu/pl/praktyczny-llm?utm_source=dwthon&utm_medium=lesson&utm_campaign=rag&utm_term=day3) moduł RAG to **5 tygodni** poświęconych dokładnie temu:

- **Hybrid search** — kiedy BM25 bije embeddingi (tak, to się zdarza częściej niż myślisz)
- **Pytania pomocnicze** — dekompozycja głównego pytania, żeby retrieval miał lepsze pokrycie
- **Ewaluacja** — jak mierzyć, czy RAG naprawdę działa dobrze (a nie tylko „wygląda, że działa")
- **Produkcja** — skalowanie, monitoring, obsługa błędów
- **Cały pipeline od PDF-a do odpowiedzi** — bez uproszczeń, pod Twoją kontrolą

Ale to już Twoja decyzja. Jeśli 3 dni wystarczyły — super, bierz kod i rozwijaj własne projekty. Jeśli czujesz, że chcesz głębiej — wiesz, gdzie szukać.

**Dzięki za te 3 dni. Do zobaczenia.** 👋